# 🎯 Integrated Salience System - End-to-End Demo

This notebook demonstrates the **fully integrated** custom salience system working with the actual memory flow!

You'll see:
- ✅ Custom salience configs **actually deciding** what to store
- ✅ Real chat interactions with decisions being made
- ✅ **Salience logs** showing WHY each decision was made
- ✅ **What got stored vs. what got skipped**
- ✅ **Compare different configs** side-by-side

This is the **REAL THING** - not a demo, but actual integration!

---

## 📦 Setup and Installation

In [ ]:
!pip install -q git+https://github.com/thebnbrkr/memlayer.git@claude/hosted-memory-service-01CkroWwJc2EpkTcQ72hS2im

import os
from memlayer import OpenAI, TenantSalienceConfig, GraphVisualizer, MemoryBrowser
from memlayer.config.salience import (
    SalienceComponent,
    ScoringFunctionType,
    AdaptiveThresholdConfig,
    ThresholdStrategy,
    DecisionRule,
)

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "your-api-key-here"  # ← REPLACE THIS

print("✅ Installation complete!")
print("="*60)

---
## 🎨 Create Custom Salience Configuration

Let's create a config that values technical content and has a decision rule!

In [ ]:
print("📊 Creating Custom Salience Configuration")
print("="*60)

# Define YOUR custom logic for what to store!
my_salience_config = TenantSalienceConfig(
    tenant_id="alice",
    config_name="tech_focused_config",
    components=[
        SalienceComponent(
            name="technical_relevance",
            weight=0.6,  # 60% of score
            description="How technical/informative is this?",
            scoring_function=ScoringFunctionType.KEYWORD_MATCH,
            scoring_config={
                "keywords": ["python", "code", "algorithm", "data", "AI", "technical", "project"]
            }
        ),
        SalienceComponent(
            name="sufficient_length",
            weight=0.4,  # 40% of score
            description="Long enough to be meaningful",
            scoring_function=ScoringFunctionType.LENGTH_BONUS,
            scoring_config={"min_length": 15, "max_length": 150}
        ),
    ],
    threshold_config=AdaptiveThresholdConfig(
        strategy=ThresholdStrategy.ABSOLUTE,
        absolute_threshold=0.5  # Need score >= 0.5 to store
    ),
    decision_rules=[
        DecisionRule(
            name="Always keep very technical messages",
            condition="technical_relevance > 0.8",
            action="STORE",
            priority=1
        ),
    ]
)

print(f"✅ Created config: {my_salience_config.config_name}")
print(f"   Components: {len(my_salience_config.components)}")
print(f"   Threshold: {my_salience_config.threshold_config.absolute_threshold}")
print(f"   Decision rules: {len(my_salience_config.decision_rules)}")
print("\n💡 This config will:")
print("   - Give 60% weight to technical keywords")
print("   - Give 40% weight to message length")
print("   - Require score >= 0.5 to store")
print("   - Always store if technical_relevance > 0.8")

---
## 🚀 Initialize Client with Custom Salience

Now let's create an OpenAI client that **uses our custom config**!

In [ ]:
print("🚀 Initializing OpenAI Client with Custom Salience Config")
print("="*60)

client = OpenAI(
    model="gpt-4o-mini",
    user_id="Alice",
    tenant_id="alice",
    storage_path="./integrated_memory",
    operation_mode="online",
    salience_config=my_salience_config,  # ← YOUR CUSTOM CONFIG!
)

print("✅ Client initialized!")
print("   Now every chat message will be scored by YOUR custom logic.")
print("   The salience calculator will decide: STORE or SKIP.")

---
## 💬 Test Chat - See Your Config in Action!

Let's send 4 different messages and see which ones YOUR config decides to store:

In [ ]:
print("💬 Testing Chat with Different Messages")
print("="*60)

test_messages = [
    "My name is Alice and I work at TechCorp.",  # Expect: STORE (long + some technical)
    "Hi!",  # Expect: SKIP (too short)
    "I'm working on a Python data analysis project using algorithms.",  # Expect: STORE (very technical!)
    "I had coffee today.",  # Expect: SKIP (not technical)
]

print("Sending 4 test messages...\n")

for i, msg in enumerate(test_messages, 1):
    print(f"[{i}/4] 📝 Sending: \"{msg}\"")
    
    response = client.chat([{"role": "user", "content": msg}])
    
    reply = response['choices'][0]['message']['content']
    print(f"     🤖 Reply: {reply[:80]}...\n")

print("✅ All messages sent!")
print("   Your custom salience config made a decision for each one.")
print("   Let's see what it decided...")

---
## 📊 Check Salience Logs - See WHY Decisions Were Made

This is the **magic** - full transparency into salience decisions!

In [ ]:
print("📊 Salience Decision Logs")
print("="*60)

logs = client.get_salience_logs()

print(f"\nFound {len(logs)} salience decisions:\n")

for i, log in enumerate(logs, 1):
    fact_preview = log['fact'][:50] + "..." if len(log['fact']) > 50 else log['fact']
    
    print(f"[Log {i}]")
    print(f"Fact: \"{fact_preview}\"")
    print(f"Decision: {'🟢 STORE' if log['decision'] == 'STORE' else '🔴 SKIP'}")
    print(f"Score: {log['score']:.3f} | Threshold: {log['threshold']:.3f}")
    
    print(f"\nComponent Breakdown:")
    for comp_name, comp_score in log['component_scores'].items():
        print(f"   - {comp_name}: {comp_score:.3f}")
    
    if log['matched_rule']:
        print(f"\n🎯 Matched Decision Rule: '{log['matched_rule']}'")
    
    print(f"💡 Reasoning: {log['reasoning']}")
    print("-"*60 + "\n")

# Summary
stored = sum(1 for log in logs if log['decision'] == 'STORE')
skipped = sum(1 for log in logs if log['decision'] == 'SKIP')

print(f"\n📈 Summary:")
print(f"   Total messages: {len(logs)}")
print(f"   Stored: {stored}")
print(f"   Skipped: {skipped}")

---
## 📦 What Actually Got Stored?

Let's check what's in the vector database:

In [ ]:
print("📦 Checking Vector Database")
print("="*60)

browser = MemoryBrowser(client.storage)
browser.show_all(user_id="Alice", max_items=20)

---
## 🕸️ Knowledge Graph Visualization

In [ ]:
print("🕸️ Knowledge Graph")
print("="*60)

viz = GraphVisualizer(client.graph_storage)
viz.print_summary()

print("\nGenerating graph visualization...")
viz.show()

---
## 🔄 Compare with a Permissive Config

Let's create a **super permissive** config that stores almost everything, and compare!

In [ ]:
print("🔄 Creating Permissive Configuration")
print("="*60)

# Super permissive config
permissive_config = TenantSalienceConfig(
    tenant_id="bob",
    config_name="store_everything",
    components=[
        SalienceComponent(
            name="always_high",
            weight=1.0,
            scoring_function=ScoringFunctionType.LENGTH_BONUS,
            scoring_config={"min_length": 1, "max_length": 10}  # Even "Hi!" scores well
        )
    ],
    threshold_config=AdaptiveThresholdConfig(
        strategy=ThresholdStrategy.ABSOLUTE,
        absolute_threshold=0.1  # Very low threshold!
    )
)

# New client
client2 = OpenAI(
    model="gpt-4o-mini",
    user_id="Bob",
    tenant_id="bob",
    storage_path="./permissive_memory",
    operation_mode="online",
    salience_config=permissive_config
)

print(f"✅ Created permissive config with threshold {permissive_config.threshold_config.absolute_threshold}")
print("   This will store almost EVERYTHING!\n")

# Test with same messages that were SKIPPED before
print("Sending messages that were SKIPPED by strict config:")
for msg in ["Hi!", "I had coffee today."]:
    print(f"\n📝 {msg}")
    client2.chat([{"role": "user", "content": msg}])

# Check logs
logs2 = client2.get_salience_logs()
print("\n" + "="*60)
print("Permissive config decisions:")
for log in logs2:
    decision_emoji = '🟢' if log['decision'] == 'STORE' else '🔴'
    print(f"{decision_emoji} {log['fact'][:40]}... → {log['decision']} (score: {log['score']:.3f})")

---
## 📊 Final Comparison

Let's see the difference between strict and permissive configs:

In [ ]:
print("📊 FINAL COMPARISON")
print("="*60)

logs1 = client.get_salience_logs()
logs2 = client2.get_salience_logs()

stored1 = sum(1 for l in logs1 if l['decision'] == 'STORE')
skipped1 = sum(1 for l in logs1 if l['decision'] == 'SKIP')

stored2 = sum(1 for l in logs2 if l['decision'] == 'STORE')
skipped2 = sum(1 for l in logs2 if l['decision'] == 'SKIP')

print("\n🎯 STRICT CONFIG (technical + length):")
print(f"   Threshold: 0.5")
print(f"   Total messages: {len(logs1)}")
print(f"   Stored: {stored1} ({stored1/len(logs1)*100:.0f}%)")
print(f"   Skipped: {skipped1} ({skipped1/len(logs1)*100:.0f}%)")

print("\n🌈 PERMISSIVE CONFIG (low threshold):")
print(f"   Threshold: 0.1")
print(f"   Total messages: {len(logs2)}")
print(f"   Stored: {stored2} ({stored2/len(logs2)*100:.0f}%)")
print(f"   Skipped: {skipped2} ({skipped2/len(logs2)*100:.0f}%)")

print("\n💡 This demonstrates:")
print("   - YOU control what gets stored")
print("   - Different configs = different behavior")
print("   - Full transparency via salience logs")
print("   - Production-ready integration!")

---
## 🎉 Summary

### What We Just Did:

✅ **Created custom salience configuration**
   - User-defined components with custom weights
   - Decision rules that override threshold
   - Complete control over storage logic

✅ **Integrated with real memory flow**
   - OpenAI client uses YOUR config
   - Every chat message scored by YOUR logic
   - Real STORE/SKIP decisions

✅ **Full transparency**
   - Salience logs show WHY decisions were made
   - Component scores for every message
   - See matched rules, thresholds, reasoning

✅ **Compared configs**
   - Strict vs. Permissive
   - Same messages, different outcomes
   - Proved flexibility!

---

## 🚀 This is PRODUCTION-READY!

The salience system is now **fully integrated** into Memlayer's memory flow.

You can:
- ✅ Deploy with ANY custom scoring logic
- ✅ Change configs without code changes
- ✅ Get full audit trail of all decisions
- ✅ Scale to multiple tenants
- ✅ Use FastAPI to manage configs

**No presets. No templates. Complete freedom. Production-ready.**

---